# Stations Pipeline (Refactored)
This notebook runs one reusable pipeline for all available station years.

In [5]:
from pathlib import Path
import pprint as pp
import pandas as pd

from generic_code.ContaminantManagerJSON import ContaminantManagerJSON
from generic_code.StationManagerJSON import StationManagerJSON
from generic_code.station_pipeline import (
    compare_years,
    compare_years_sequence,
    condense_station_rows,
    discover_station_years,
    load_station_year,
    normalize_station_columns,
    run_station_pipeline,
    summarize_station_df,
)

In [6]:
STATIONS_FOLDER = Path('../data/bronze/stations')
CONTAMINANTS_JSON = Path('../data/gold/contaminants/contaminants.json')
MAP_OUTPUT_FOLDER = Path('./interative_maps')
TARGET_YEARS = discover_station_years(STATIONS_FOLDER)
TARGET_YEARS

[2018, 2019, 2021, 2022, 2023, 2025, 2026]

## Simplified step-by-step station EDA
This section reproduces the core ideas of the pipeline in smaller, easier-to-debug steps.

In [7]:
# Step 1: discover available station years
years_available = discover_station_years(STATIONS_FOLDER)
print("Years available:", years_available)

# Use a subset first to make step-by-step analysis faster
years_for_step_by_step = years_available[:3]
print("Years selected for step-by-step EDA:", years_for_step_by_step)

Years available: [2018, 2019, 2021, 2022, 2023, 2025, 2026]
Years selected for step-by-step EDA: [2018, 2019, 2021]


In [8]:
# Step 2: load one year and inspect raw data
sample_year_step = years_for_step_by_step[0]
raw_df_step = load_station_year(STATIONS_FOLDER, sample_year_step)

print(f"Raw shape for {sample_year_step}:", raw_df_step.shape)
print("Raw columns:")
pp.pprint(raw_df_step.columns.tolist())
raw_df_step.head()

Raw shape for 2018: (7, 16)
Raw columns:
['nom_cabina',
 'codi_dtes',
 'zqa',
 'codi_eoi',
 'longitud',
 'latitud',
 'Ubicacio',
 'Codi_Districte',
 'Nom_Districte',
 'Codi_Barri',
 'Nom_Barri',
 'Ocupacio_sol',
 'Emissions_Properes',
 'Contaminant_1',
 'Contaminant_2',
 'Contaminant_3']


,nom_cabina,codi_dtes,zqa,codi_eoi,longitud,latitud,Ubicacio,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Ocupacio_sol,Emissions_Properes,Contaminant_1,Contaminant_2,Contaminant_3
0,Barcelona - Ciutadella,IL,1,8019050,2.1874,41.3864,Parc de la Ciutadella,1,Ciutat Vella,4,"Sant Pere, Santa Caterina i la Ribera",Urbana,Fons,NO2,O3,NaN
1,Barcelona - Eixample,IH,1,8019043,2.1538,41.3853,Av. Roma - c/ Comte Urgell,5,Eixample,9,la Nova Esquerra de l'Eixample,Urbana,Trànsit,NO2,O3,PM10
2,Barcelona - Gràcia,IJ,1,8019044,2.1534,41.3987,Plaça Gal·la Placídia (Via Augusta - Travesser...,6,Gracia,31,la Vila de Gracia,Urbana,Trànsit,NO2,O3,PM10
3,Barcelona - Palau Reial,IZ,1,8019057,2.1151,41.3875,c/ John Maynard Keynes - c/ de Jordi Girona,4,Les Corts,21,Pedralbes,Urbana,Fons,NO2,O3,PM10
4,Barcelona - Poblenou,I2,1,8019004,2.2045,41.4039,Plaça Josep Trueta (Pujades - Lope de Vega),10,Sant Marti,68,el Poblenou,Urbana,Fons,NO2,NaN,PM10


In [9]:
# Step 3: normalize columns and compute a compact EDA summary
normalized_df_step = normalize_station_columns(raw_df_step)
summary_step = summarize_station_df(normalized_df_step)

print("Normalized columns:")
pp.pprint(normalized_df_step.columns.tolist())
print("\nSummary:")
pp.pprint(summary_step)

normalized_df_step.head()

Normalized columns:
['station_name',
 'station_code',
 'aqzc',
 'eoi_code',
 'longitud',
 'latitud',
 'Ubicacio',
 'Codi_Districte',
 'Nom_Districte',
 'Codi_Barri',
 'Nom_Barri',
 'Ocupacio_sol',
 'Emissions_Properes',
 'Contaminant_1',
 'Contaminant_2',
 'Contaminant_3']

Summary:
{'columns': ['station_name',
             'station_code',
             'aqzc',
             'eoi_code',
             'longitud',
             'latitud',
             'Ubicacio',
             'Codi_Districte',
             'Nom_Districte',
             'Codi_Barri',
             'Nom_Barri',
             'Ocupacio_sol',
             'Emissions_Properes',
             'Contaminant_1',
             'Contaminant_2',
             'Contaminant_3'],
 'duplicate_rows': 0,
 'missing_values': {'Codi_Barri': 0,
                    'Codi_Districte': 0,
                    'Contaminant_1': 0,
                    'Contaminant_2': 2,
                    'Contaminant_3': 2,
                    'Emissions_Properes': 0,
    

,station_name,station_code,aqzc,eoi_code,longitud,latitud,Ubicacio,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,Ocupacio_sol,Emissions_Properes,Contaminant_1,Contaminant_2,Contaminant_3
0,Barcelona - Ciutadella,IL,1,8019050,2.1874,41.3864,Parc de la Ciutadella,1,Ciutat Vella,4,"Sant Pere, Santa Caterina i la Ribera",Urbana,Fons,NO2,O3,NaN
1,Barcelona - Eixample,IH,1,8019043,2.1538,41.3853,Av. Roma - c/ Comte Urgell,5,Eixample,9,la Nova Esquerra de l'Eixample,Urbana,Trànsit,NO2,O3,PM10
2,Barcelona - Gràcia,IJ,1,8019044,2.1534,41.3987,Plaça Gal·la Placídia (Via Augusta - Travesser...,6,Gracia,31,la Vila de Gracia,Urbana,Trànsit,NO2,O3,PM10
3,Barcelona - Palau Reial,IZ,1,8019057,2.1151,41.3875,c/ John Maynard Keynes - c/ de Jordi Girona,4,Les Corts,21,Pedralbes,Urbana,Fons,NO2,O3,PM10
4,Barcelona - Poblenou,I2,1,8019004,2.2045,41.4039,Plaça Josep Trueta (Pujades - Lope de Vega),10,Sant Marti,68,el Poblenou,Urbana,Fons,NO2,NaN,PM10


In [10]:
# Step 4: create ContaminantManagerJSON and condense one year
contaminant_manager_step = ContaminantManagerJSON(str(CONTAMINANTS_JSON))
condensed_df_step = condense_station_rows(normalized_df_step, contaminant_manager_step)

print(f"Condensed shape for {sample_year_step}:", condensed_df_step.shape)
condensed_df_step.head()

ValueError: Cannot condense station data. Missing required columns: contaminant_code

In [ ]:
# Step 5: repeat simplified EDA for a few years and build in-memory export data
yearly_step_data = {}
export_data_step = {}

for y in years_for_step_by_step:
    df_raw = load_station_year(STATIONS_FOLDER, y)
    df_norm = normalize_station_columns(df_raw)
    df_cond = condense_station_rows(df_norm, contaminant_manager_step)
    summary = summarize_station_df(df_norm)

    yearly_step_data[y] = {
        "raw": df_raw,
        "normalized": df_norm,
        "condensed": df_cond,
        "summary": summary,
    }

    station_map = {}
    for _, row in df_cond.iterrows():
        station_id = str(row["station_code"])
        station_map[station_id] = {
            col: row[col] for col in df_cond.columns if col != "station_code"
        }

    export_data_step[str(y)] = station_map

summary_table = pd.DataFrame(
    [
        {
            "year": y,
            "shape": yearly_step_data[y]["summary"]["shape"],
            "stations_count": yearly_step_data[y]["summary"].get("stations_count"),
            "contaminants_count": yearly_step_data[y]["summary"].get("contaminants_count"),
            "duplicate_rows": yearly_step_data[y]["summary"]["duplicate_rows"],
        }
        for y in years_for_step_by_step
    ]
)
summary_table

## Step-by-step StationManagerJSON tests
Run the next cells in order to test each method individually.

In [ ]:
# Test 1: initialize from simplified export data
station_manager_step = StationManagerJSON.from_pipeline_export_data(
    export_data_step,
    contaminant_manager=contaminant_manager_step,
)
station_manager_step

In [ ]:
# Test 2: years and station ids
print("Available years:", station_manager_step.get_available_years())
print("Has year", years_for_step_by_step[0], ":", station_manager_step.has_year(years_for_step_by_step[0]))
print("All station ids (union across years):", station_manager_step.get_station_ids())
print("Station ids in first selected year:", station_manager_step.get_station_ids(years_for_step_by_step[0]))

In [ ]:
# Test 3: station existence and station data retrieval
test_year = years_for_step_by_step[0]
test_station = station_manager_step.get_station_ids(test_year)[0]

print("Test station:", test_station)
print("Has station in selected year:", station_manager_step.has_station(test_station, test_year))
print("Has station in any year:", station_manager_step.has_station(test_station))
print("Latest station data:")
pp.pprint(station_manager_step.get_latest_station_data(test_station))
print("Station data in selected year:")
pp.pprint(station_manager_step.get_station_data(test_station, test_year))

In [ ]:
# Test 4: contaminants measured by station
codes = station_manager_step.get_station_contaminant_codes(test_station, test_year)
print("Contaminant codes:", codes)

metadata_list = station_manager_step.get_station_contaminant_metadata(test_station, test_year)
print("Contaminant metadata summaries:")
pp.pprint(metadata_list)

In [ ]:
# Test 5: reverse lookup - stations measuring a contaminant in a year
test_code = codes[0] if codes else 10
stations_with_code = station_manager_step.get_stations_measuring_contaminant(test_code, test_year)
print(f"Stations measuring contaminant code {test_code} in {test_year}:", stations_with_code)

In [ ]:
# Test 6: full in-memory data snapshot
all_station_data_step = station_manager_step.get_all_data()
print("Years in all data:", list(all_station_data_step.keys()))
print("Sample year station count:", len(all_station_data_step[str(test_year)]))

In [4]:
pipeline_result = run_station_pipeline(
    stations_folder=STATIONS_FOLDER,
    contaminants_json=CONTAMINANTS_JSON,
    years=TARGET_YEARS,
    save_maps=True,
    map_output_folder=MAP_OUTPUT_FOLDER,
)

ValueError: Cannot condense station data. Missing required columns: contaminant_code

In [ ]:
for year in pipeline_result['years']:
    summary = pipeline_result['yearly_results'][year].summary
    print(f'Year {year}:')
    print(f"  Shape: {summary['shape']}")
    print(f"  Stations: {summary.get('stations_count', 'N/A')}")
    print(f"  Contaminants: {summary.get('contaminants_count', 'N/A')}")
    print(f"  Duplicate rows: {summary['duplicate_rows']}")
    print('-' * 60)

In [ ]:
# JSON-ready dictionary: {'2019': {'station_code': {...}}, ...}
export_data = pipeline_result['export_data']
list(export_data.keys())

In [ ]:
# Example: condensed table for one year
pipeline_result['yearly_results'][TARGET_YEARS[0]].condensed_df.head()

## Compare years
Use these cells to see station and contaminant changes between years.

In [ ]:
# Example 1: compare two specific years
comparison_2019_2021 = compare_years(
    year_a=2019,
    year_b=2021,
    yearly_results=pipeline_result['yearly_results'],
    contaminants_json=CONTAMINANTS_JSON,
)
comparison_2019_2021

In [ ]:
# Example 2: compare each consecutive pair of years
sequence_comparisons = compare_years_sequence(
    years=pipeline_result['years'],
    yearly_results=pipeline_result['yearly_results'],
    contaminants_json=CONTAMINANTS_JSON,
)
sequence_comparisons

## Use StationManagerJSON with ContaminantManagerJSON
These cells keep station data in memory and let you query contaminants measured in a station.

In [ ]:
contaminant_manager = ContaminantManagerJSON(str(CONTAMINANTS_JSON))
station_manager = StationManagerJSON.from_pipeline_export_data(
    export_data,
    contaminant_manager=contaminant_manager,
)

station_manager

In [ ]:
sample_year = pipeline_result['years'][0]
sample_station = station_manager.get_station_ids(sample_year)[0]

print(f"Sample year: {sample_year}")
print(f"Sample station: {sample_station}")
print("\nStation data:")
print(station_manager.get_station_data(sample_station, sample_year))

print("\nContaminant metadata measured in this station:")
station_manager.get_station_contaminant_metadata(sample_station, sample_year)